In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
from pathlib import Path

project_path = Path("/content/drive/MyDrive/UM_WQF7023/MRDRP-main")

exposures_dir = project_path / "exposures"
outcome_dir = project_path / "outcome"

target_record_file = project_path / "artesunate_file_screening_record.csv"

exposures_dir.mkdir(exist_ok=True)
outcome_dir.mkdir(exist_ok=True)

print("Project path:", project_path)
print("Exposures folder:", exposures_dir, exposures_dir.exists())
print("Outcome folder:", outcome_dir, outcome_dir.exists())
print("Target record file:", target_record_file)

Project path: /content/drive/MyDrive/UM_WQF7023/MRDRP-main
Exposures folder: /content/drive/MyDrive/UM_WQF7023/MRDRP-main/exposures True
Outcome folder: /content/drive/MyDrive/UM_WQF7023/MRDRP-main/outcome True
Target record file: /content/drive/MyDrive/UM_WQF7023/MRDRP-main/artesunate_file_screening_record.csv


In [ ]:
import shutil
import pandas as pd
from pathlib import Path


def infer_sep(file_name):
    """Infer delimiter from file extension."""
    if file_name.endswith(".csv") or file_name.endswith(".csv.gz"):
        return ","
    return "\t"


def infer_compression(file_name):
    """Infer compression type from file extension."""
    if file_name.endswith(".gz"):
        return "gzip"
    return None


def is_supported_file(file_name):
    """Check whether the file format is supported."""
    return (
        file_name.endswith(".tsv.gz")
        or file_name.endswith(".tsv")
        or file_name.endswith(".csv.gz")
        or file_name.endswith(".csv")
    )


def read_preview_safely(file_path, nrows=5):
    """
    Read only the first few rows of a GWAS file.
    If Google Drive reading fails, copy the file to /content first and retry.
    """
    file_path = Path(file_path)
    sep = infer_sep(file_path.name)
    compression = infer_compression(file_path.name)

    try:
        df_small = pd.read_csv(
            file_path,
            sep=sep,
            compression=compression,
            nrows=nrows
        )
        return df_small

    except OSError:
        # Fallback for Google Drive transport issues
        local_path = Path("/content") / file_path.name
        shutil.copy2(file_path, local_path)

        df_small = pd.read_csv(
            local_path,
            sep=sep,
            compression=compression,
            nrows=nrows
        )
        return df_small


def inspect_columns(columns):
    """
    Inspect whether the file contains MR-related fields.
    """

    expected_groups = {
        "variant_identifier": ["rsid", "SNP", "variant_id", "hm_rsid", "rs_number", "variant"],
        "chromosome": ["chromosome", "chrom", "CHR", "chr"],
        "position": ["base_pair_location", "position", "POS", "pos"],
        "effect_allele": ["effect_allele", "EA", "A1", "alt"],
        "other_allele": ["other_allele", "OA", "A2", "ref"],
        "beta": ["beta", "BETA", "effect", "estimate", "odds_ratio", "OR", "or", "odds ratio", "log_odds", "lnOR"],
        "standard_error": ["standard_error", "SE", "se"],
        "effect_allele_frequency": ["effect_allele_frequency", "EAF", "eaf", "af"],
        "p_value": ["p_value", "P", "p", "pval"],
        "sample_size": ["n", "N", "sample_size", "samplesize"]
    }

    found_map = {}

    for group, candidates in expected_groups.items():
        found = [col for col in candidates if col in columns]
        found_map[group] = found[0] if len(found) > 0 else None

    core_fields = [
        "chromosome",
        "position",
        "effect_allele",
        "other_allele",
        "beta",
        "standard_error",
        "effect_allele_frequency",
        "p_value"
    ]

    missing_core = [field for field in core_fields if found_map[field] is None]

    if len(missing_core) == 0:
        full_summary_stats_candidate = "Yes"

        if found_map["variant_identifier"] is not None:
            status = "Suitable candidate"
            notes = "Core MR fields found; variant identifier present."
        else:
            status = "Needs adaptation"
            notes = "Core MR fields found, but no obvious rsID/SNP-like identifier column."
    else:
        full_summary_stats_candidate = "No"
        status = "Not suitable for current pipeline"
        notes = f"Missing core fields: {', '.join(missing_core)}"

    return found_map, full_summary_stats_candidate, status, notes

In [ ]:
def scan_role_folder(base_dir, role):
    """
    Scan files from one role folder.

    Expected structure:
        exposures/IGF1/file.tsv.gz
        exposures/SHBG/file.tsv.gz
        exposures/ADIPOQ/file.tsv.gz
        exposures/GDF15/file.tsv.gz

        outcome/endometrial_cancer/file.tsv.gz

    The subfolder name will be used as the trait label.
    """

    base_dir = Path(base_dir)
    records = []

    if not base_dir.exists():
        print(f"Folder not found: {base_dir}")
        return records

    for trait_folder in sorted(base_dir.iterdir()):
        if not trait_folder.is_dir():
            continue

        trait_name = trait_folder.name

        for file_path in sorted(trait_folder.iterdir()):
            if not file_path.is_file():
                continue

            if not is_supported_file(file_path.name):
                continue

            print("=" * 100)
            print("Scanning:", file_path)
            print("Role:", role)
            print("Trait folder:", trait_name)

            try:
                df_small = read_preview_safely(file_path, nrows=5)
                columns = df_small.columns.tolist()

                found_map, full_summary_stats_candidate, status, notes = inspect_columns(columns)

                record = {
                    "file_name": file_path.name,
                    "file_path": str(file_path),
                    "trait": trait_name,
                    "study_accession": file_path.name
                        .replace(".tsv.gz", "")
                        .replace(".tsv", "")
                        .replace(".csv.gz", "")
                        .replace(".csv", ""),
                    "role": role,
                    "build": "Unknown",
                    "variant_identifier": found_map["variant_identifier"],
                    "chromosome": found_map["chromosome"],
                    "position": found_map["position"],
                    "effect_allele": found_map["effect_allele"],
                    "other_allele": found_map["other_allele"],
                    "beta": found_map["beta"],
                    "standard_error": found_map["standard_error"],
                    "effect_allele_frequency": found_map["effect_allele_frequency"],
                    "p_value": found_map["p_value"],
                    "sample_size": found_map["sample_size"],
                    "full_summary_stats_candidate": full_summary_stats_candidate,
                    "status": status,
                    "notes": notes
                }

                records.append(record)

                print("Status:", status)
                print("Detected columns:", columns)

            except Exception as e:
                print("Error processing file:", file_path)
                print(e)

    return records


# Scan exposure and outcome folders
exposure_records = scan_role_folder(exposures_dir, role="exposure")
outcome_records = scan_role_folder(outcome_dir, role="outcome")

# Combine records
all_records = exposure_records + outcome_records

# This is the target_df that later cells will use
target_df = pd.DataFrame(all_records)

print("\nTotal records generated:", len(target_df))

if len(target_df) > 0:
    display(target_df)
else:
    print("No supported files found.")

Scanning: /content/drive/MyDrive/UM_WQF7023/MRDRP-main/exposures/ADIPOQ/GCST90100621_buildGRCh37.tsv.gz
Role: exposure
Trait folder: ADIPOQ
Status: Not suitable for current pipeline
Detected columns: ['chromosome', 'base_pair_location', 'effect_allele', 'other_allele', 'beta', 'effect_allele_frequency', 'p_value', 'z', 'neff', 'model', 'rs_id', 'SEQID']
Scanning: /content/drive/MyDrive/UM_WQF7023/MRDRP-main/exposures/ADIPOQ/GCST90161837_buildGRCh37.tsv.gz
Role: exposure
Trait folder: ADIPOQ
Status: Suitable candidate
Detected columns: ['p_value', 'chromosome', 'base_pair_location', 'variant_id', 'effect_allele', 'other_allele', 'effect_allele_frequency', 'beta', 'standard_error']
Scanning: /content/drive/MyDrive/UM_WQF7023/MRDRP-main/exposures/ADIPOQ/GCST90237429_buildGRCh38.tsv.gz
Role: exposure
Trait folder: ADIPOQ
Status: Suitable candidate
Detected columns: ['chromosome', 'base_pair_location', 'other_allele', 'effect_allele', 'beta', 'standard_error', 'effect_allele_frequency', 'Qu

,file_name,file_path,trait,study_accession,role,build,variant_identifier,chromosome,position,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,sample_size,full_summary_stats_candidate,status,notes
0,GCST90100621_buildGRCh37.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,ADIPOQ,GCST90100621_buildGRCh37,exposure,Unknown,None,chromosome,base_pair_location,effect_allele,other_allele,beta,None,effect_allele_frequency,p_value,None,No,Not suitable for current pipeline,Missing core fields: standard_error
1,GCST90161837_buildGRCh37.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,ADIPOQ,GCST90161837_buildGRCh37,exposure,Unknown,variant_id,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,None,Yes,Suitable candidate,Core MR fields found; variant identifier present.
2,GCST90237429_buildGRCh38.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,ADIPOQ,GCST90237429_buildGRCh38,exposure,Unknown,variant_id,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,None,Yes,Suitable candidate,Core MR fields found; variant identifier present.
3,demo_ADIPOQ_buildGRCh38.csv,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,ADIPOQ,demo_ADIPOQ_buildGRCh38,exposure,Unknown,rsid,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,sample_size,Yes,Suitable candidate,Core MR fields found; variant identifier present.
4,GCST90162057_buildGRCh37.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,GDF15,GCST90162057_buildGRCh37,exposure,Unknown,variant_id,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,None,Yes,Suitable candidate,Core MR fields found; variant identifier present.
5,GCST90179306_buildGRCh38.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,GDF15,GCST90179306_buildGRCh38,exposure,Unknown,None,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,None,Yes,Needs adaptation,"Core MR fields found, but no obvious rsID/SNP-..."
6,demo_GDF15_buildGRCh38.csv,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,GDF15,demo_GDF15_buildGRCh38,exposure,Unknown,rsid,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,sample_size,Yes,Suitable candidate,Core MR fields found; variant identifier present.
7,GCST90014008_buildGRCh38.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,IGF1,GCST90014008_buildGRCh38,exposure,Unknown,None,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,None,p_value,None,No,Not suitable for current pipeline,Missing core fields: effect_allele_frequency
8,GCST90019511_buildGRCh37.tsv,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,IGF1,GCST90019511_buildGRCh37,exposure,Unknown,variant_id,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,None,p_value,None,No,Not suitable for current pipeline,Missing core fields: effect_allele_frequency
9,GCST90025989_buildGRCh37.tsv,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,IGF1,GCST90025989_buildGRCh37,exposure,Unknown,None,chromosome,base_pair_location,None,None,beta,standard_error,None,p_value,None,No,Not suitable for current pipeline,"Missing core fields: effect_allele, other_alle..."


In [ ]:
if "target_df" not in globals():
    raise NameError("target_df is not defined. Please run Cell 3 first.")

if len(target_df) > 0:
    target_df = target_df.drop_duplicates(subset=["file_path"], keep="last")
    target_df.to_csv(target_record_file, index=False)

    print("Saved Artesunate screening record to:")
    print(target_record_file)

    display(target_df)
else:
    print("target_df is empty. No CSV file was created.")

Saved Artesunate screening record to:
/content/drive/MyDrive/UM_WQF7023/MRDRP-main/artesunate_file_screening_record.csv


,file_name,file_path,trait,study_accession,role,build,variant_identifier,chromosome,position,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,sample_size,full_summary_stats_candidate,status,notes
0,GCST90100621_buildGRCh37.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,ADIPOQ,GCST90100621_buildGRCh37,exposure,Unknown,None,chromosome,base_pair_location,effect_allele,other_allele,beta,None,effect_allele_frequency,p_value,None,No,Not suitable for current pipeline,Missing core fields: standard_error
1,GCST90161837_buildGRCh37.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,ADIPOQ,GCST90161837_buildGRCh37,exposure,Unknown,variant_id,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,None,Yes,Suitable candidate,Core MR fields found; variant identifier present.
2,GCST90237429_buildGRCh38.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,ADIPOQ,GCST90237429_buildGRCh38,exposure,Unknown,variant_id,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,None,Yes,Suitable candidate,Core MR fields found; variant identifier present.
3,demo_ADIPOQ_buildGRCh38.csv,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,ADIPOQ,demo_ADIPOQ_buildGRCh38,exposure,Unknown,rsid,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,sample_size,Yes,Suitable candidate,Core MR fields found; variant identifier present.
4,GCST90162057_buildGRCh37.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,GDF15,GCST90162057_buildGRCh37,exposure,Unknown,variant_id,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,None,Yes,Suitable candidate,Core MR fields found; variant identifier present.
5,GCST90179306_buildGRCh38.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,GDF15,GCST90179306_buildGRCh38,exposure,Unknown,None,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,None,Yes,Needs adaptation,"Core MR fields found, but no obvious rsID/SNP-..."
6,demo_GDF15_buildGRCh38.csv,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,GDF15,demo_GDF15_buildGRCh38,exposure,Unknown,rsid,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,sample_size,Yes,Suitable candidate,Core MR fields found; variant identifier present.
7,GCST90014008_buildGRCh38.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,IGF1,GCST90014008_buildGRCh38,exposure,Unknown,None,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,None,p_value,None,No,Not suitable for current pipeline,Missing core fields: effect_allele_frequency
8,GCST90019511_buildGRCh37.tsv,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,IGF1,GCST90019511_buildGRCh37,exposure,Unknown,variant_id,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,None,p_value,None,No,Not suitable for current pipeline,Missing core fields: effect_allele_frequency
9,GCST90025989_buildGRCh37.tsv,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,IGF1,GCST90025989_buildGRCh37,exposure,Unknown,None,chromosome,base_pair_location,None,None,beta,standard_error,None,p_value,None,No,Not suitable for current pipeline,"Missing core fields: effect_allele, other_alle..."


In [ ]:
if target_record_file.exists():
    saved_df = pd.read_csv(target_record_file)
    print("Loaded saved record:")
    print("Shape:", saved_df.shape)
    display(saved_df)
else:
    print("Record file does not exist yet.")

Loaded saved record:
Shape: (36, 19)


,file_name,file_path,trait,study_accession,role,build,variant_identifier,chromosome,position,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,sample_size,full_summary_stats_candidate,status,notes
0,GCST90100621_buildGRCh37.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,ADIPOQ,GCST90100621_buildGRCh37,exposure,Unknown,NaN,chromosome,base_pair_location,effect_allele,other_allele,beta,NaN,effect_allele_frequency,p_value,NaN,No,Not suitable for current pipeline,Missing core fields: standard_error
1,GCST90161837_buildGRCh37.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,ADIPOQ,GCST90161837_buildGRCh37,exposure,Unknown,variant_id,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,NaN,Yes,Suitable candidate,Core MR fields found; variant identifier present.
2,GCST90237429_buildGRCh38.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,ADIPOQ,GCST90237429_buildGRCh38,exposure,Unknown,variant_id,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,NaN,Yes,Suitable candidate,Core MR fields found; variant identifier present.
3,demo_ADIPOQ_buildGRCh38.csv,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,ADIPOQ,demo_ADIPOQ_buildGRCh38,exposure,Unknown,rsid,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,sample_size,Yes,Suitable candidate,Core MR fields found; variant identifier present.
4,GCST90162057_buildGRCh37.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,GDF15,GCST90162057_buildGRCh37,exposure,Unknown,variant_id,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,NaN,Yes,Suitable candidate,Core MR fields found; variant identifier present.
5,GCST90179306_buildGRCh38.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,GDF15,GCST90179306_buildGRCh38,exposure,Unknown,NaN,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,NaN,Yes,Needs adaptation,"Core MR fields found, but no obvious rsID/SNP-..."
6,demo_GDF15_buildGRCh38.csv,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,GDF15,demo_GDF15_buildGRCh38,exposure,Unknown,rsid,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,p_value,sample_size,Yes,Suitable candidate,Core MR fields found; variant identifier present.
7,GCST90014008_buildGRCh38.tsv.gz,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,IGF1,GCST90014008_buildGRCh38,exposure,Unknown,NaN,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,NaN,p_value,NaN,No,Not suitable for current pipeline,Missing core fields: effect_allele_frequency
8,GCST90019511_buildGRCh37.tsv,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,IGF1,GCST90019511_buildGRCh37,exposure,Unknown,variant_id,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,NaN,p_value,NaN,No,Not suitable for current pipeline,Missing core fields: effect_allele_frequency
9,GCST90025989_buildGRCh37.tsv,/content/drive/MyDrive/UM_WQF7023/MRDRP-main/e...,IGF1,GCST90025989_buildGRCh37,exposure,Unknown,NaN,chromosome,base_pair_location,NaN,NaN,beta,standard_error,NaN,p_value,NaN,No,Not suitable for current pipeline,"Missing core fields: effect_allele, other_alle..."
